# Module 04 — Cell Classification

This notebook summarises the per-dataset binary cell classification produced by
`scripts/04_annotation.py`. Each of the **11 datasets** (392,107 cells total,
excluding GSE233666) was independently classified into two classes:

1. **Mesenchymal** — IVD resident cells (NP, AF, EP lineages) identified by
   expression of mesenchymal markers (COL2A1, ACAN, SOX9, COL1A1, etc.)
2. **Non-mesenchymal** — immune, endothelial, and pericyte populations identified
   by non-mesenchymal markers (PTPRC/CD45, PECAM1, CD68, etc.)

Classification uses `sc.tl.score_genes()` to compute mesenchymal and
non-mesenchymal signature scores per cell, followed by thresholding. The result
is stored in the `cell_class` column of each processed `.h5ad` file.

Detailed cell type annotation (NP subtypes, AF subtypes, immune subtypes, etc.)
is deferred to **Module 05**, where it is performed after cross-dataset
integration with scVI and Leiden clustering.

**Data sources:**
- `data/processed/{accession}.h5ad` — annotated AnnData objects (11 files, with `cell_class` in `.obs`)
- GSE233666 is excluded in v2 of the pipeline

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='scanpy')

from pathlib import Path

import anndata
anndata.settings.allow_write_nullable_strings = True

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path('..').resolve()
PROC_DIR = BASE / 'data' / 'processed'
ANN_DIR = BASE / 'results' / 'annotations'
ANN_DIR.mkdir(parents=True, exist_ok=True)

# GSE233666 excluded in v2
ALL_ACCESSIONS = [
    'GSE160756', 'GSE165722', 'GSE189916', 'GSE199866', 'GSE205535',
    'CNP0002664', 'GSE244889', 'GSE251686', 'GSE255768',
    'GSE230809', 'GSE242443',
]

PALETTE = dict(zip(ALL_ACCESSIONS, sns.color_palette('tab20', len(ALL_ACCESSIONS))))

# ── Load obs DataFrames (no .X) ──────────────────────────────────────────
obs_frames = {}
for acc in ALL_ACCESSIONS:
    path = PROC_DIR / f'{acc}.h5ad'
    if not path.exists():
        print(f'  SKIP {acc}: file not found')
        continue
    adata = sc.read_h5ad(path, backed='r')
    obs_frames[acc] = adata.obs.to_memory() if hasattr(adata.obs, 'to_memory') else adata.obs.copy()
    adata.file.close()
    print(f'  {acc}: {len(obs_frames[acc]):,} cells')

print(f'\nLoaded {len(obs_frames)} datasets')

## Summary Table

The table below shows the mesenchymal / non-mesenchymal cell counts per dataset.
Each dataset was independently classified using signature scoring — mesenchymal
cells express IVD resident markers (collagens, proteoglycans, SOX9), while
non-mesenchymal cells express immune/endothelial markers (PTPRC, PECAM1, etc.).

In [ ]:
# Collect cell class counts per dataset
ct_records = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    obs = obs_frames[acc]
    if 'cell_class' not in obs.columns:
        print(f'  WARNING: {acc} missing cell_class column')
        continue
    counts = obs['cell_class'].value_counts()
    for cc, n in counts.items():
        ct_records.append({'accession': acc, 'cell_class': cc, 'count': n})

ct_df = pd.DataFrame(ct_records)

# Pivot table: datasets x cell classes
pivot = ct_df.pivot_table(index='accession', columns='cell_class', values='count',
                          fill_value=0, aggfunc='sum')
pivot = pivot.loc[[a for a in ALL_ACCESSIONS if a in pivot.index]]

# Add totals
pivot['TOTAL'] = pivot.sum(axis=1)

# Add percentage columns
for col in [c for c in pivot.columns if c != 'TOTAL']:
    pivot[f'{col} (%)'] = (pivot[col] / pivot['TOTAL'] * 100).round(1)

display(pivot.style.set_caption('Cell Class Counts per Dataset (Mesenchymal vs Non-Mesenchymal)'))

## UMAP Grid — Cell Classification per Dataset

Each panel shows one dataset. The UMAP is computed per-dataset (before
integration) and colored by the binary `cell_class` label (mesenchymal vs
non-mesenchymal). This gives a visual overview of the mesenchymal/non-mesenchymal
split in each study.

In [ ]:
# Colour palette for cell classes
class_palette = {'mesenchymal': '#2ecc71', 'non_mesenchymal': '#e74c3c'}

# Load UMAP embeddings
umap_data = {}
for acc in ALL_ACCESSIONS:
    path = PROC_DIR / f'{acc}.h5ad'
    if not path.exists() or acc not in obs_frames:
        continue
    if 'cell_class' not in obs_frames[acc].columns:
        continue
    adata = sc.read_h5ad(path, backed='r')
    if 'X_umap' in adata.obsm:
        umap_data[acc] = adata.obsm['X_umap'][:]
    adata.file.close()

datasets = [a for a in ALL_ACCESSIONS if a in umap_data]
n_datasets = len(datasets)

fig, axes = plt.subplots(n_datasets, 1, figsize=(10, 4 * n_datasets))
if n_datasets == 1:
    axes = [axes]

for ax, acc in zip(axes, datasets):
    umap = umap_data[acc]
    labels = obs_frames[acc]['cell_class'].values
    idx = np.random.RandomState(42).permutation(len(umap))
    colors = [class_palette.get(str(labels[i]), '#cccccc') for i in idx]
    ax.scatter(umap[idx, 0], umap[idx, 1], c=colors, s=0.5, alpha=0.6, rasterized=True)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_ylabel(acc, fontsize=10, fontweight='bold')
    # Add cell counts as text
    n_mes = (labels == 'mesenchymal').sum()
    n_non = (labels == 'non_mesenchymal').sum()
    ax.text(0.98, 0.95, f'mes: {n_mes:,}\nnon-mes: {n_non:,}',
            transform=ax.transAxes, fontsize=7, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Legend at the bottom
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=class_palette[t],
                   markersize=8, label=t) for t in ['mesenchymal', 'non_mesenchymal']]
fig.legend(handles=handles, loc='lower center', ncol=2,
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Per-Dataset UMAP — Cell Classification (Mesenchymal vs Non-Mesenchymal)',
             fontsize=14, y=1.0)
fig.tight_layout()
fig.savefig(ANN_DIR / 'notebook_04_umap_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## Marker Expression — Mesenchymal vs Non-Mesenchymal

Aggregated across all datasets: for each cell class, we show the expression of
key discriminating markers. Mesenchymal cells should express IVD resident markers
(COL2A1, ACAN, SOX9, COL1A1, etc.) while non-mesenchymal cells should express
immune and endothelial markers (PTPRC, PECAM1, CD68, VWF, etc.).

In [ ]:
# For the dot plot we need expression data — load one dataset at a time and
# compute mean expression per cell class, then pool.
MARKER_GENES = [
    # Mesenchymal / IVD resident markers
    'COL2A1', 'ACAN', 'SOX9', 'COL1A1', 'COL1A2',
    'KRT19', 'KRT8', 'KRT18', 'CD24', 'NOG',
    'THY1', 'DCN', 'LUM', 'COMP', 'PRG4',
    # Non-mesenchymal markers
    'PTPRC', 'PECAM1', 'VWF', 'CD68', 'CD3D',
    'CD79A', 'ACTA2', 'MYH11',
]

# Pool mean expression across datasets
mean_records = []
frac_records = []
for acc in ALL_ACCESSIONS:
    path = PROC_DIR / f'{acc}.h5ad'
    if acc not in obs_frames or 'cell_class' not in obs_frames[acc].columns:
        continue
    adata = sc.read_h5ad(path)
    available = [g for g in MARKER_GENES if g in adata.var_names]
    if not available:
        del adata; continue
    sub = adata[:, available]
    from scipy import sparse
    X = sub.X.toarray() if sparse.issparse(sub.X) else np.asarray(sub.X)
    cc_labels = obs_frames[acc]['cell_class'].values
    for cc in np.unique(cc_labels):
        mask = cc_labels == cc
        expr = X[mask]
        mean_expr = expr.mean(axis=0)
        frac_expr = (expr > 0).mean(axis=0)
        for j, gene in enumerate(available):
            mean_records.append({'cell_class': cc, 'gene': gene,
                                 'mean_expr': mean_expr[j], 'n_cells': mask.sum()})
            frac_records.append({'cell_class': cc, 'gene': gene,
                                 'frac_expr': frac_expr[j], 'n_cells': mask.sum()})
    del adata, sub, X

# Weighted average across datasets
mean_df = pd.DataFrame(mean_records)
frac_df = pd.DataFrame(frac_records)
if len(mean_df) > 0:
    mean_agg = mean_df.groupby(['cell_class', 'gene']).apply(
        lambda g: np.average(g['mean_expr'], weights=g['n_cells'])
    ).reset_index(name='mean_expr')
    frac_agg = frac_df.groupby(['cell_class', 'gene']).apply(
        lambda g: np.average(g['frac_expr'], weights=g['n_cells'])
    ).reset_index(name='frac_expr')

    # Pivot for heatmap
    mean_pivot = mean_agg.pivot(index='cell_class', columns='gene', values='mean_expr').fillna(0)
    frac_pivot = frac_agg.pivot(index='cell_class', columns='gene', values='frac_expr').fillna(0)

    # Reorder genes
    gene_order = [g for g in MARKER_GENES if g in mean_pivot.columns]
    mean_pivot = mean_pivot[gene_order]
    frac_pivot = frac_pivot[gene_order]

    # Custom dot plot
    fig, ax = plt.subplots(figsize=(max(14, len(gene_order) * 0.6), 4))
    cell_classes_sorted = sorted(mean_pivot.index.tolist())
    for i, cc in enumerate(cell_classes_sorted):
        for j, gene in enumerate(gene_order):
            size = frac_pivot.loc[cc, gene] * 200
            color_val = mean_pivot.loc[cc, gene]
            ax.scatter(j, i, s=size, c=color_val, cmap='Reds', vmin=0,
                       vmax=mean_pivot.values.max(), edgecolors='grey', linewidth=0.3)

    ax.set_xticks(range(len(gene_order)))
    ax.set_xticklabels(gene_order, rotation=90, fontsize=8)
    ax.set_yticks(range(len(cell_classes_sorted)))
    ax.set_yticklabels(cell_classes_sorted, fontsize=10)
    ax.set_title('Marker Expression: Mesenchymal vs Non-Mesenchymal (Aggregated)', fontsize=11)
    ax.set_xlabel('Marker gene')
    ax.set_ylabel('Cell class')

    # Add vertical line separating mesenchymal from non-mesenchymal markers
    ax.axvline(x=14.5, color='grey', linestyle='--', alpha=0.5)
    ax.text(7, -0.7, 'Mesenchymal markers', ha='center', fontsize=8, color='grey')
    ax.text(18, -0.7, 'Non-mesenchymal', ha='center', fontsize=8, color='grey')

    fig.tight_layout()
    fig.savefig(ANN_DIR / 'notebook_04_dotplot.png', dpi=150)
    plt.show()
else:
    print('No expression data available for dotplot')

## Stacked Bar — Cell Class Proportions per Dataset

The stacked bar chart shows the relative proportions of mesenchymal vs
non-mesenchymal cells within each dataset. Differences reflect tissue
compartment (NP-enriched vs AF-enriched vs mixed), disease state, and
technical differences in cell capture across studies.

In [ ]:
# Build proportion matrix
prop_records = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames or 'cell_class' not in obs_frames[acc].columns:
        continue
    obs = obs_frames[acc]
    n = len(obs)
    counts = obs['cell_class'].value_counts()
    for cc, cnt in counts.items():
        prop_records.append({'accession': acc, 'cell_class': cc, 'proportion': cnt / n})

prop_df = pd.DataFrame(prop_records)
prop_pivot = prop_df.pivot_table(index='accession', columns='cell_class',
                                  values='proportion', fill_value=0)
prop_pivot = prop_pivot.loc[[a for a in ALL_ACCESSIONS if a in prop_pivot.index]]

# Colours
bar_colors = {'mesenchymal': '#2ecc71', 'non_mesenchymal': '#e74c3c'}
col_order = [c for c in ['mesenchymal', 'non_mesenchymal'] if c in prop_pivot.columns]
prop_pivot = prop_pivot[col_order]

fig, ax = plt.subplots(figsize=(14, 6))
prop_pivot.plot.bar(stacked=True, ax=ax,
                    color=[bar_colors[c] for c in col_order],
                    width=0.75, edgecolor='white')
ax.set_ylabel('Proportion')
ax.set_xlabel('')
ax.set_title('Cell Class Proportions per Dataset')
ax.tick_params(axis='x', rotation=45)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9, frameon=False)
ax.set_ylim(0, 1)

fig.tight_layout()
fig.savefig(ANN_DIR / 'notebook_04_proportions.png', dpi=150, bbox_inches='tight')
plt.show()

## Classification Score Distributions

The mesenchymal and non-mesenchymal signature scores (computed via
`sc.tl.score_genes()`) are shown for each cell class. Well-separated
distributions indicate confident classification. Overlapping distributions
suggest ambiguous cells near the decision boundary.

In [ ]:
# Pool mesenchymal/non-mesenchymal scores across datasets
score_cols = ['score_mesenchymal', 'score_non_mesenchymal']
score_records = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    obs = obs_frames[acc]
    available = [c for c in score_cols if c in obs.columns]
    if not available or 'cell_class' not in obs.columns:
        continue
    sub = obs[available + ['cell_class']].copy()
    sub['accession'] = acc
    score_records.append(sub)

if score_records:
    scores_df = pd.concat(score_records, ignore_index=True)
    available_score_cols = [c for c in score_cols if c in scores_df.columns]

    fig, axes = plt.subplots(1, len(available_score_cols),
                              figsize=(6 * len(available_score_cols), 5))
    if len(available_score_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, available_score_cols):
        for cc in ['mesenchymal', 'non_mesenchymal']:
            data = scores_df.loc[scores_df['cell_class'] == cc, col].dropna()
            if len(data) > 10:
                data.plot.kde(ax=ax, label=cc, alpha=0.7,
                             color={'mesenchymal': '#2ecc71', 'non_mesenchymal': '#e74c3c'}[cc])
        ax.set_title(col.replace('score_', '').replace('_', ' ').title(), fontsize=11)
        ax.set_xlabel('Score')
        ax.legend(fontsize=9)

    fig.suptitle('Classification Scores by Cell Class', fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(ANN_DIR / 'notebook_04_score_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No classification scores available')

## Label Transition — Preliminary to Cell Class

This heatmap shows how cells flow from the preliminary coarse labels
(Module 03: broad types based on a few markers each) to the binary cell
class labels (Module 04: mesenchymal vs non-mesenchymal). This validates
that the classification correctly groups NP/AF/EP preliminary types as
mesenchymal and immune/endothelial types as non-mesenchymal.

In [ ]:
# Collect preliminary -> cell_class mappings
flow_records = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    obs = obs_frames[acc]
    if 'cell_type_preliminary' not in obs.columns or 'cell_class' not in obs.columns:
        continue
    for (prel, cc), n in obs.groupby(['cell_type_preliminary', 'cell_class']).size().items():
        flow_records.append({'preliminary': prel, 'cell_class': cc, 'count': n})

if flow_records:
    flow_df = pd.DataFrame(flow_records)
    flow_agg = flow_df.groupby(['preliminary', 'cell_class'])['count'].sum().reset_index()

    transition = flow_agg.pivot_table(index='preliminary', columns='cell_class',
                                       values='count', fill_value=0)
    # Normalize by row
    transition_norm = transition.div(transition.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(8, max(5, len(transition_norm) * 0.5)))
    sns.heatmap(transition_norm, cmap='Blues', annot=True, fmt='.2f', ax=ax,
                linewidths=0.5, cbar_kws={'label': 'Fraction of preliminary type'})
    ax.set_xlabel('Cell class')
    ax.set_ylabel('Preliminary label (Module 03)')
    ax.set_title('Preliminary Label -> Cell Class Mapping (Row-Normalized)')

    fig.tight_layout()
    fig.savefig(ANN_DIR / 'notebook_04_label_transition.png', dpi=150)
    plt.show()
else:
    print('Preliminary labels not available for comparison')

## Notes and Next Steps

**Classification parameters:**
- Binary classification: mesenchymal vs non-mesenchymal
- Gene signatures scored with `sc.tl.score_genes()` (mesenchymal: COL2A1, ACAN,
  SOX9, COL1A1, etc.; non-mesenchymal: PTPRC, PECAM1, CD68, etc.)
- Classification stored in `cell_class` column of each processed `.h5ad` file
- GSE233666 excluded from v2 pipeline (11 datasets remain)

**What happens next (Module 05):**
- Cross-dataset integration with scVI per compartment (NP, AF, CEP)
- Leiden clustering on integrated embeddings
- De novo cell type annotation with expanded IVD-specific signatures
- CellTypist validation for non-mesenchymal subtypes
- Result: detailed `cell_type` labels (NP_mature_chondrocyte, AF_inner, T_cell, etc.)

**Figures saved** to `results/annotations/` with the `notebook_04_` prefix.

In [ ]:
saved_figures = sorted(ANN_DIR.glob('notebook_04_*.png'))
print('Saved figures:')
for f in saved_figures:
    print(f'  {f.relative_to(BASE)}')